In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as f

spark = SparkSession \
    .builder \
    .appName("BDLC_Parking_Violations_Analysis") \
    .master("spark://bdlc-012.bdlc.ls.eee.intern:7077") \
    .getOrCreate()

spark

In [ ]:
%%time
processed_path = "hdfs:///parking_violations/processed/parking_violations_cleaned"

df = spark.read.parquet(processed_path)
df.cache()
df.count()

In [ ]:
df.printSchema()

## Analyse 1: Häufigste Parking Violation Typen FY2023–FY2025
Welche Violation-Typen kommen am häufigsten vor und wie verändern sie sich über die Jahre?

In [ ]:
%%time
violation_by_year = df.groupBy("fiscal_year", "violation_code", "violation_description") \
    .count() \
    .orderBy("fiscal_year", f.desc("count"))

violation_by_year.show(20, truncate=False)

### Resultate
Violation Code 36 (Schulzone Tempolimit) dominiert mit über 8 Mio. Verstössen in FY2023 
deutlich. Code 21 (Parkverbot Strassenreinigung) und Code 38 (Parkuhr) folgen auf 
Platz 2 und 3.

### Visualisierung Top 10
Die häufigsten Violation-Typen über alle drei Fiskaljahre (2023–2025) zusammengefasst.

In [ ]:
# Top 10 Violation Codes gesamt
top10 = df.groupBy("violation_code", "violation_description") \
    .count() \
    .orderBy(f.desc("count")) \
    .limit(10) \
    .toPandas()

top10.plot(
    x="violation_description",
    y="count",
    kind="barh",
    figsize=(12, 6),
    title="Top 10 Parking Violations FY2023–FY2025",
    xlabel="Anzahl Violations",
    legend=False
)

### Trend über die Jahre
Verändern sich die häufigsten Violations über FY2023, FY2024 und FY2025?

In [ ]:
# Top 5 Codes pro Jahr
top5_codes = ["36", "21", "38", "14", "7"]

yearly_trend = df.filter(f.col("violation_code").isin(top5_codes)) \
    .groupBy("fiscal_year", "violation_code") \
    .count() \
    .orderBy("violation_code", "fiscal_year") \
    .toPandas()

yearly_trend.pivot(index="fiscal_year", columns="violation_code", values="count") \
    .plot(kind="bar", figsize=(12, 6), title="Top 5 Violation Codes pro Jahr")

### Detailzahlen Code 36
Um die Veränderung über die Jahre genau zu beziffern, schauen wir uns 
die absoluten Zahlen für den häufigsten Verstoss an.

In [ ]:
%%time
yearly_counts = df.filter(f.col("violation_code") == "36") \
    .groupBy("fiscal_year") \
    .count() \
    .orderBy("fiscal_year") \
    .toPandas()

print("Code 36 - PHTO SCHOOL ZN SPEED VIOLATION:")
print(yearly_counts.to_string(index=False))

### Detailzahlen Top 5 Codes
Übersicht aller Top-5 Violation Codes mit exakten Zahlen pro Fiskaljahr.

In [ ]:
summary = df.filter(f.col("violation_code").isin(["36", "21", "38", "14", "7"])) \
    .groupBy("fiscal_year", "violation_code", "violation_description") \
    .count() \
    .orderBy("violation_code", "fiscal_year") \
    .toPandas()

print(summary.to_string(index=False))

### Anmerkung zu "Unknown"
Einige Einträge zeigen `Unknown` als `violation_description`. Diese stammen aus 
dem Pre-Processing, wo fehlende Beschreibungsfelder mit `Unknown` ersetzt wurden. 
Die Anzahl ist vernachlässigbar klein (< 0.5% pro Code) und beeinflusst die 
Analyse nicht. Für die weitere Auswertung werden diese Einträge gefiltert.

In [ ]:
%%time
summary_clean = df.filter(f.col("violation_code").isin(["36", "21", "38", "14", "7"])) \
    .filter(f.col("violation_description") != "Unknown") \
    .groupBy("fiscal_year", "violation_code") \
    .count() \
    .orderBy("violation_code", "fiscal_year") \
    .toPandas()

print(summary_clean.to_string(index=False))

### Interpretation
- Code 36 (Schulzone Tempolimit) ist mit Abstand der häufigste Verstoss, 
  sinkt aber von 8 Mio. (FY2023) auf 4.9 Mio. (FY2025) — ein Rückgang von ~39%.
- Code 21 (Parkverbot Strassenreinigung) bleibt stabil auf Platz 2.
- Alle Top-5 Codes zeigen einen leichten Rückgang über die drei Jahre.
- Die Reihenfolge der häufigsten Violations bleibt über alle Jahre konstant.

### Fazit Analyse 1
Die Frage "Welche Violation-Typen kommen am häufigsten vor?" kann klar beantwortet werden:

Code 36 (Schulzone Tempolimit) dominiert mit grossem Abstand — dies ist womöglich auf
automatische Fotoblitzer in Schulzonen zurückzuführen. Bemerkenswert ist der 
kontinuierliche Rückgang über alle Top-Codes von FY2023 auf FY2025, was auf 
weniger Verstösse oder veränderte Enforcement-Strategien hindeuten könnte.

In [ ]:
spark.stop()